# CNN training
This notebook recreates the four-class CNN used by `trained_model.h5`. It uses the same 224×224 RGB conversion, Gaussian blur, and 0–1 scaling as the FastAPI service.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TRAINING_DIR = PROJECT / 'Training'
TESTING_DIR = PROJECT / 'Testing'
OUTPUT_DIR = PROJECT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
IMG_SIZE = (224, 224)
CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
CLASS_DIRECTORIES = ['glioma', 'meningioma', 'notumor', 'pituitary']

def blur_image(image):
    return cv2.GaussianBlur(image, (3, 3), 0)

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(*IMG_SIZE, 3)),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(), Dense(128, activation='relu'), Dropout(0.4),
    Dense(len(CLASS_NAMES), activation='softmax'),
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
datagen = ImageDataGenerator(
    rescale=1/255, validation_split=0.2, preprocessing_function=blur_image,
    rotation_range=30, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=0.2, fill_mode='nearest'
)
train_data = datagen.flow_from_directory(TRAINING_DIR, target_size=IMG_SIZE, batch_size=32,
    class_mode='categorical', subset='training', classes=CLASS_DIRECTORIES)
validation_data = datagen.flow_from_directory(TRAINING_DIR, target_size=IMG_SIZE, batch_size=32,
    class_mode='categorical', subset='validation', classes=CLASS_DIRECTORIES)
assert train_data.class_indices == {name: i for i, name in enumerate(CLASS_DIRECTORIES)}
history = model.fit(train_data, epochs=20, validation_data=validation_data)

In [ ]:
test_data = ImageDataGenerator(rescale=1/255, preprocessing_function=blur_image).flow_from_directory(
    TESTING_DIR, target_size=IMG_SIZE, batch_size=32, class_mode='categorical',
    shuffle=False, classes=CLASS_DIRECTORIES)
loss, accuracy = model.evaluate(test_data)
model.save(OUTPUT_DIR / 'trained_model.keras')
print(f'Test accuracy: {accuracy:.2%}')